# Exploratory analysis — biomedical-rag-bench

The **exploratory** notebook (the scrappy first of two). Its job is to *de-risk* the polished
notebook: first establish the data is trustworthy, then sketch the headline figures. It imports
the one tested loader (`eval/analysis/load.py`) — no load / dedup / reshape logic lives in cells.

Two steps, in this order on purpose:
- **Step 1 — the data: grain, dimensions, checks (*no charts*).** Can I *trust* this? Each
  dimension is shown with its realized value and linked to its canonical explanation in the READMEs.
- **Step 2 — metrics & insights (*charts*).** What does it *say*?

Run with the `eval` extra: `uv run --extra eval jupyter lab eval/analysis/explore.ipynb`.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parents[1]))  # repo root — this notebook lives in eval/analysis/

import pandas as pd
from eval.analysis.load import load

df = load()
print(f"{len(df)} canonical rows · {df['retriever'].nunique()} retrievers · {df['type_id'].nunique()} types")

## Step 1 — The data: grain, dimensions, checks (no charts)

No plots here. We state the **coordinate system** of the analysis — its *grain*, *dimensions*,
and *measures* — each shown with its realized value and linked to the canonical explanation, and
we *assert* the data matches what we claim. In pandas terms: **dimensions = groupby keys,
measures = agg targets, granularity = which keys you group on.** Step 2 then plots within it.

### Grain — one row = one trial

The atomic unit is **one `(question × retriever × generator)` trial → a verdict**. Everything
downstream aggregates over it. The canonical dedup grain is `(retriever, generator_model_family,
question_id)`. → [full result-row schema](../README.md#result-row-schema)

In [ ]:
key = ["retriever", "generator_model_family", "question_id"]
assert not df.duplicated(key).any(), "grain not unique — dedup bug in load.py"
print("grain unique on", key)
df[["question_id", "retriever", "generator_model", "passed"]].head(3)

### Dimensions, by role

A factorial experiment has factors that are *varied* (under test), *held fixed* (for
comparability), and *confounded* (entangled — read with care). Naming the role up front is what
lets Step 2 say "we slice by type, we don't marginalize over it" and have it land.

**`retriever`** — *varied*; the condition under test (the swap point).
→ [Roster + per-retriever](../../retrievers/README.md#roster)

In [ ]:
df.groupby("retriever")["question_id"].nunique().rename("n_questions").sort_values(ascending=False)

**`type_id`** — *varied*; the 10-type hop/structure taxonomy the H1 crossover is about.
→ [Question type taxonomy](../README.md#question-type-taxonomy)

In [ ]:
df.groupby("type_id")["question_id"].nunique().rename("n_questions")

**`generator`** — *held fixed within a run*, varied across runs. `generator_model` is the
*requested* id (may be a moving alias); `generator_model_resolved` is the *exact snapshot* the
provider ran. → [configured vs. resolved id](../generate/README.md#the-model-under-test--configured-vs-resolved-id)

In [ ]:
df[["generator_model", "generator_model_resolved", "generator_model_family"]].drop_duplicates().reset_index(drop=True)

**`scoring`** — selects which judge/metric applies; **tied to `type_id`** (not independent).
→ [Strategy → judge map](../judge/README.md#strategy--judge-map)

In [ ]:
ct = pd.crosstab(df["type_id"], df["scoring"])
print("each type maps to exactly one scoring strategy:", bool((ct.gt(0).sum(axis=1) == 1).all()))
ct

### Confounds (named up front)

- **`type_id` ⟷ `scoring`** — each type has exactly one judge (crosstab above), so "by type" and
  "by scoring" are the *same* cut, not two independent dimensions.
- **hops ⟷ fan-caps** — raising graph hops without raising the fan caps *buries* the answer; a
  coupled knob, not a clean factor. → [FINDINGS caveat](../FINDINGS.md)
- **writer-LLM inside `graph_sparqlgen`** — that arm runs an LLM *inside* retrieval; its token
  cost is a *mechanism* cost, logged separately and **never** summed with the generator's billed
  tokens. → [graph_sparqlgen](../../retrievers/README.md#roster)

### Measures & granularities

**Measures:** `passed` → accuracy (binary) · `recall` / `precision` / `f1` (the richer picture) ·
`retrieval_token_premium` + writer/judge LLM cost (*never summed*) · latency.
→ [Metrics](../README.md#metrics) · [token-units rule](../../retrievers/README.md#the-token-units-rule-read-before-doing-any-token-math)

**Granularities:** per-trial → per `(retriever × type)` **cell** (the heatmap) → per-retriever
**marginal** (*the misleading top-line — we slice, we don't marginalize*) → per-type marginal.

### Telemetry coverage — what's analyzable now vs. after the re-runs

The new retriever-telemetry columns are `NaN` for runs made before the harness persisted them, so
the writer-cost / cosine / top_k panels stay empty until the canonical re-runs land. This table is
the honest "what can I compute today" line.

In [ ]:
cols = ["recall", "retrieval_token_premium", "writer_input_tokens",
        "sparql_valid", "top_k", "hops", "cache_read_input_tokens"]
cov = pd.DataFrame({"non_null": [int(df[c].notna().sum()) if c in df else 0 for c in cols]},
                   index=cols)
cov["total"] = len(df)
cov["pct"] = (100 * cov.non_null / cov.total).round(0)
cov

### Reconciliation against FINDINGS (the trust linchpin)

Does the loader reproduce the hand-recorded **deterministic** pass counts in `eval/FINDINGS.md`?
A disagreement means a loader bug — catch it *here*, before any chart inherits it. (We filter out
`semantic`, whose 6 type-10 questions are counted separately and judged by an uncalibrated LLM.)

In [ ]:
det = df[df["scoring"] != "semantic"]
got = det.groupby("retriever")["passed"].sum().astype(int)
expected = {"closed_book": 4, "vector": 8, "graph_neighborhood_1hop": 13,
            "graph_neighborhood_2hop": 12, "graph_sparqlgen": 15}
rec = pd.DataFrame({"loader": got}).join(pd.Series(expected, name="FINDINGS"))
rec["match"] = rec["loader"] == rec["FINDINGS"]
rec  # NaN FINDINGS = a deprecated/smoke condition not in the recorded set

### Integrity checks

In [ ]:
print("error rows:", int(df["error"].notna().sum()) if "error" in df else 0)
print("unjudged non-semantic rows:", int(((~df["judged"]) & (df["scoring"] != "semantic")).sum()))
print("max generators per run (should be 1):", int(df.groupby("run_id")["generator_model"].nunique().max()))

## Step 2 — Metrics & insights (charts)

The frame is trusted, so we plot *within* the coordinate system Step 1 defined. Cost/writer panels
are `NaN` until the canonical re-runs on the new telemetry schema (see coverage above) — they light
up automatically when those land, since the notebook just re-imports `load()`.

### Fig 1 — Accuracy heatmap (retriever × type)

The spine of the whole analysis: the crossover (H1), the structural-type wall, the 0-hop tie. We
*slice* by type — never read the per-retriever marginal as the headline.

In [ ]:
import matplotlib.pyplot as plt

det = df[df["scoring"] != "semantic"]
piv = det.pivot_table(index="retriever", columns="type_id", values="passed", aggfunc="mean")
fig, ax = plt.subplots(figsize=(12, 3.5))
im = ax.imshow(piv.values, cmap="RdYlGn", vmin=0, vmax=1, aspect="auto")
ax.set_xticks(range(len(piv.columns))); ax.set_xticklabels(piv.columns, rotation=45, ha="right")
ax.set_yticks(range(len(piv.index))); ax.set_yticklabels(piv.index)
for i in range(piv.shape[0]):
    for j in range(piv.shape[1]):
        v = piv.values[i, j]
        if pd.notna(v):
            ax.text(j, i, f"{v:.1f}", ha="center", va="center", fontsize=8)
fig.colorbar(im, label="accuracy"); ax.set_title("Accuracy by retriever × type (deterministic)")
plt.tight_layout()

### Fig 2 — Recall vs. exact-set pass (the `graph_sparqlgen` finding)

The set/aggregate judges pass only on an *exact* set; **recall** reveals how often the *complete*
answer was actually retrieved even when the binary verdict failed (precision leaks sink it).

In [ ]:
sm = df[(df["scoring"] == "set_match") & df["recall"].notna()]
g = sm.groupby("retriever").agg(mean_recall=("recall", "mean"),
                                exact_pass_rate=("passed", "mean"),
                                n=("passed", "size"))
ax = g[["mean_recall", "exact_pass_rate"]].plot.bar(figsize=(9, 4))
ax.set_ylabel("rate"); ax.set_ylim(0, 1)
ax.set_title("set_match: mean recall vs. exact-set pass rate (divergence = retrieved-but-failed)")
plt.tight_layout()
g.round(2)

### Fig 3 — Precision leak on set questions

Where recall is high but extra rows sink F1 — `graph_sparqlgen`'s underconstrained queries return
supersets (worst on set-difference / 3+hop).

In [ ]:
df[df["scoring"] == "set_match"].groupby("retriever").agg(
    recall=("recall", "mean"), precision=("precision", "mean"),
    f1=("f1", "mean"), mean_extra=("num_extra", "mean")).round(2)

### Cost — retrieval token premium (writer cost pending re-run)

`retrieval_token_premium` = billed `input_tokens` − closed_book input for the same question+model
(the one unit-safe decomposition). `writer_in` is the `graph_sparqlgen` writer-LLM cost — `NaN`
until the re-runs, demonstrating the coverage gap rather than hiding it.

In [ ]:
df.groupby("retriever").agg(
    premium=("retrieval_token_premium", "mean"),
    writer_in=("writer_input_tokens", "mean")).round(1)

### Next

- **Canonical re-runs** on the new telemetry schema populate the writer / top_k / cache / cosine
  columns and the full-corpus `vector` arm — these cells then refresh automatically.
- The **polished notebook** imports the same `load()` and refines these into the definitive figures
  (and adds the type-08 sensitivity panel and the type-10 *uncalibrated* note).